# ITSPM: Colab quickstart
Choose **Runtime → Change runtime type → GPU**, then run the setup and small examples.
These examples check the pipeline; they do not reproduce full benchmark scores.
The default data are bundled or generated locally. P12, PAM, USHCN, Stocks and Energy need separate preparation; see the [data guide](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/DATASETS.md).


In [ ]:
%cd /content
!git clone https://github.com/zhouziyu02/IP_ITSPM.git
%cd /content/IP_ITSPM
%pip install -q -r requirements.txt


In [ ]:
import torch
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!python3 scripts/check_data.py --dataset bundled --verify


## 1. Small real-data forecasting run
Activity is already bundled. This uses eight source records, one epoch and a small model.
Training/validation/test identities are split before window expansion. The full preset is `Extrapolation/scripts/activity.sh`.


In [ ]:
!python3 regression.py --model ITSPM --dataset activity --task forecasting --history 3000 -n 8 --epoch 1 --batch_size 8 --d_model 16 --n_ref_points 8 --n_scales 2 --n_mixer_layers 1 --max_event_tokens 16 --max_gap_tokens 8 --seed 0 --state colab_quickstart


## 2. Small generation run
Sines needs no external download. Two epochs and ten discriminator steps are only a functionality check, not a useful quality estimate.


In [ ]:
!SEQ_LENS=12 MISSING_RATIOS=0.5 N_SAMPLES=256 DS_ITERATIONS=10 GPU=0 SEED=0 bash Generation/scripts/sines.sh --ae_epochs 2 --diff_epochs 2 --batch_size 16


## 3. CPU regression checks
This covers all task families, including classification on synthetic fixtures and a real Taxi batch.


In [ ]:
!bash scripts/run_checks.sh


## Continue with your project
Read [README](https://github.com/zhouziyu02/IP_ITSPM), [project milestones](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/PROJECTS.md), and [reference-result limitations](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/REFERENCE_RESULTS.md).

After preparing the chosen dataset, run its preset with explicit seeds 0, 1 and 2, keep splits and budgets fixed, and report mean ± sample standard deviation. Trailing flags such as `--epoch 5 --batch_size 8` override classification/regression presets. The generation head still needs a predictive-score evaluator for project 1.5; projects 1.6–1.8 require extensions.

Checkpoints, results and logs are local and ignored by Git. Copy them to your own persistent storage before the Colab runtime disconnects; mounted Google Drive is one option. Do not treat a short run or historical single-seed result as a completed reproduction.
